In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display
from io import StringIO


def visualize_dataset_summary(summary: dict, save_pdf: str | None = None) -> str:
    """
    Visualize a dataset summary in Jupyter, save a full PDF report, and return a formatted docstring.

    Parameters
    ----------
    summary : dict
        Dictionary from summarize_dataset() with keys such as
        'duration_stats', 'duration_histogram', 'signal_coverage',
        'dataset_classes', 'class_distribution', 'extra_class_distribution'.
    save_pdf : str or None
        If provided, saves a complete multi-page PDF report to this path.
    figsize : tuple
        Figure size for plots.

    Returns
    -------
    str
        A formatted Python docstring summarizing the dataset.
    """
    n_patients = summary.get("n_patients", None)
    dur_stats = summary.get("duration_stats", {})
    dur_hist = summary.get("duration_histogram", {})
    signal_cov = summary.get("signal_coverage", pd.DataFrame())
    dataset_classes = summary.get("dataset_classes", None)
    class_dist = summary.get("class_distribution", None)
    extra_class_dist = summary.get("extra_class_distribution", None)

    # -------------------------------------------------------
    # 1️⃣ Construct docstring
    # -------------------------------------------------------
    buf = StringIO()
    buf.write('"""Dataset Summary\n')
    buf.write("Summary of core statistics for this dataset.\n\n")
    
    if n_patients:
        buf.write(f"n_patients: {n_patients} \n")

    if dur_stats:
        buf.write("Durations:\n")
        for k, v in dur_stats.items():
            buf.write(f"    {k:<8}: {v}\n")

    if isinstance(signal_cov, pd.DataFrame) and not signal_cov.empty:
        buf.write("\nChannels:\n")
        for i, (ch, row) in enumerate(signal_cov.iterrows(), 1):
            buf.write(f"    {i:2d}. {ch:<20} {row['coverage[%]']:.1f}%\n")

    if dataset_classes:
        buf.write("\nClasses:\n")
        for c in dataset_classes:
            buf.write(f"    - {c}\n")

    if class_dist:
        buf.write("\nClass Frequencies:\n")
        for k, v in sorted(class_dist.items(), key=lambda x: x[1], reverse=True):
            buf.write(f"    {k:<15}: {v}\n")

    if extra_class_dist:
        buf.write("\nExtra Class Frequencies:\n")
        for k, v in sorted(extra_class_dist.items(), key=lambda x: x[1], reverse=True):
            buf.write(f"    {k:<15}: {v}\n")

    buf.write('"""\n')
    docstring = buf.getvalue()

    # -------------------------------------------------------
    # 2️⃣ Display inline in Jupyter
    # -------------------------------------------------------

    if n_patients:
        print(f"🥼Number of patients: {n_patients} \n")
    
    print("⏱ Duration Statistics")
    if dur_stats:
        display(pd.DataFrame([dur_stats]).T.rename(columns={0: "Value"}))
    else:
        print("No duration stats available.\n")

    if isinstance(signal_cov, pd.DataFrame) and not signal_cov.empty:
        print("\n📡 Channel Coverage")
        display(signal_cov)

    if dataset_classes:
        print("\n📚 Declared Dataset Classes:")
        display(pd.DataFrame(dataset_classes, columns=["Class"]))

    if class_dist:
        print("\n📊 Event Class Distribution")
        df = pd.DataFrame.from_dict(class_dist, orient="index", columns=["count"]).sort_values("count", ascending=False)
        display(df)

    if extra_class_dist:
        print("\n🔍 Extra Event Class Distribution")
        df_extra = pd.DataFrame.from_dict(extra_class_dist, orient="index", columns=["count"]).sort_values("count", ascending=False)
        display(df_extra)

    # -------------------------------------------------------
    # 3️⃣ PDF: include text, tables, plots
    # -------------------------------------------------------
    pdf = PdfPages(save_pdf) if save_pdf else None

    def show_and_save(fig):
        """Display the figure in Jupyter and save it to the PDF."""
        if pdf:
            pdf.savefig(fig, bbox_inches="tight")
        display(fig)
        plt.close(fig)

    # --- Cover page ---
    if pdf:
        fig, ax = plt.subplots(figsize=(8.3, 11.7))
        ax.axis("off")
        ax.text(0.05, 0.95, "Dataset Summary", fontsize=18, weight="bold", va="top")
        ax.text(0.05, 0.90, docstring, fontsize=9, family="monospace", va="top")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

    # --- Duration histogram ---
    if dur_hist:
        fig, ax = plt.subplots(figsize=(10, 4))
        x, y = list(dur_hist.keys()), list(dur_hist.values())
        ax.bar(x, y, color="#4C72B0")
        ax.set_xticklabels(x, rotation=45, ha="right")
        ax.set_ylabel("Count")
        ax.set_title("Recording Duration Distribution")
        fig.tight_layout()
        show_and_save(fig)

    # --- Signal coverage ---
    if isinstance(signal_cov, pd.DataFrame) and not signal_cov.empty:
        top_cov = signal_cov.sort_values("coverage[%]", ascending=False)
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.barh(top_cov.index, top_cov["coverage[%]"], color="#55A868")
        ax.invert_yaxis()
        ax.set_xlabel("Coverage [% of files]")
        ax.set_title("Most Frequent Channels")
        fig.tight_layout()
        show_and_save(fig)

    # --- Class distributions ---
    if class_dist:
        df = pd.DataFrame.from_dict(class_dist, orient="index", columns=["count"]).sort_values("count", ascending=False)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(df.index, df["count"], color="#C44E52")
        ax.set_xticklabels(df.index, rotation=45, ha="right")
        ax.set_ylabel("Count")
        ax.set_title("Event Class Distribution")
        fig.tight_layout()
        show_and_save(fig)

    # --- Extra class distributions ---
    if extra_class_dist:
        df_extra = pd.DataFrame.from_dict(extra_class_dist, orient="index", columns=["count"]).sort_values("count", ascending=False)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(df_extra.index, df_extra["count"], color="#8172B3")
        ax.set_xticklabels(df_extra.index, rotation=45, ha="right")
        ax.set_ylabel("Count")
        ax.set_title("Extra Event Class Distribution")
        fig.tight_layout()
        show_and_save(fig)

    if pdf:
        pdf.close()
        print(f"📄 Saved full summary PDF to: {save_pdf}")

    return docstring

In [ ]:
from sleepwalker.datasets.CAP import CAP
from sleepwalker.datasets.MNC import MNC
from sleepwalker.datasets.NCHSDB import NCHSDB
from sleepwalker.datasets.SHHS import SHHS
from sleepwalker.datasets.ABC import ABC
from sleepwalker.datasets.ISRUC import ISRUC
from sleepwalker.datasets.Ruhrlandklinik import Ruhrlandklinik
from sleepwalker.datasets.SleepEDFx import SleepEDFx
from sleepwalker.datasets.SVUH_UCD import SVUH_UCD
from sleepwalker.datasets.Apples import Apples
from sleepwalker.datasets.HCHS import HCHS
from sleepwalker.datasets.MROS import MROS
from sleepwalker.datasets.Numom2b import Numom2b
from sleepwalker.datasets.WSC import WSC
from sleepwalker.datasets.Stages import Stages

from sleepwalker.datasets.utils import get_edf_files_in_repo, summarize_dataset
from IPython.display import Markdown, display

# import logging
# from sleepwalker.utils import logger
# logger.set_level(logging.ERROR)

datasets = [
    # ("/cephfs_projects/sleepwalker/abc", ".edf", "C4", ABC),
    # ("/cephfs_projects/sleepwalker/apples/polysomnography", ".edf", "EMG", Apples),
    # ("/cephfs_projects/sleepwalker/cap", ".edf", "Fp2-F4", CAP),
    # ("/cephfs_projects/sleepwalker/isruc", ".edf", "E1-M2", ISRUC),
    # ("/cephfs_projects/sleepwalker/mnc/cnc", ".edf", "E2", MNC),
    # ("/raid/sleepwalker/mros", ".edf", "C4", MROS),
    # ("/cephfs_projects/sleepwalker/mros", ".edf", "C4", MROS),
    # ("/cephfs_projects/sleepwalker/ruhrlandklinik/raw", ".edf", "C4", Ruhrlandklinik),
    ("/raid/sleepwalker/shhs", ".edf", "EMG", SHHS),
    # ("/cephfs_projects/sleepwalker/sleep-edfx", "-PSG.edf", "EEG Fpz-Cz", SleepEDFx),
    # ("/cephfs_projects/sleepwalker/svuh-ucd", ".rec", "EMG", SVUH_UCD),
    # ("/cephfs_projects/sleepwalker/hchs", ".edf", "A-Snore", HCHS),
    # ("/cephfs_projects/sleepwalker/nchsdb/sleep_data", ".edf", "EEG F4-M1", NCHSDB),
    # ("/cephfs_projects/sleepwalker/numom2b", ".edf", "ECG", Numom2b),
    # ("/cephfs_projects/sleepwalker/stages", ".edf", "C4", Stages),
    # ("/cephfs_projects/sleepwalker/wsc", ".edf", "ECG", WSC)
]

for edf_path, ending, channel_name, clazz in datasets:
    clazz_name = f"{clazz}".split(".")[-1].replace(">","").replace("'","")
    display(Markdown(f"## 💤 **{clazz_name}**"))
    display(Markdown("---"))

    edf_files = get_edf_files_in_repo(edf_path, ending=ending, recursive=True)
    summary = summarize_dataset(clazz, edf_files, channel_name=channel_name, estimate_class_frequencies=False)

    pdf_path = f"./{clazz_name}_summary.pdf"
    docstring = visualize_dataset_summary(summary, save_pdf=pdf_path)

    display(Markdown("**📘 Auto-generated Docstring:**"))
    print(docstring)
    display(Markdown(f"✅ Summary saved for **{clazz_name}** → `{pdf_path}`"))
    display(Markdown("---"))



## 💤 **SHHS**

---

2025-10-17 13:24:25,984 | INFO | Preparing labels and sliding windows:   6%|▌         | 495/8444 [01:45<27:45,  4.77it/s]

2025-10-17 13:26:11,726 | INFO | Found a total of 10481 EDF files


2025-10-17 13:24:25,984 | INFO | Preparing labels and sliding windows:   6%|▌         | 495/8444 [01:48<27:45,  4.77it/s]

...

2025-10-17 13:50:49,002 | WARNING | utils.py:337 | Traceback (most recent call last):
  File "/root/projects/sleepwalker/sleepwalker/datasets/Basedataset.py", line 251, in prepare_patient
    df_additional = self.get_extra_event_df(edf_path, start)
  File "/root/projects/sleepwalker/sleepwalker/datasets/SHHS.py", line 63, in get_extra_event_df
    df = read_xml(fpath, "profusion", self.trim_wake, start_date)
  File "/root/projects/sleepwalker/sleepwalker/datasets/SHHS.py", line 18, in read_xml
    xml_df = read_profusion(xml_path)
  File "/root/projects/sleepwalker/sleepwalker/datasets/utils.py", line 201, in read_profusion
    def read_profusion(xml_path):
FileNotFoundError: [Errno 2] No such file or directory: '/raid/sleepwalker/shhs/polysomnography/annotations-events-profusion/shhs1/shhs1-204591-profusion.xml'



2025-10-17 13:51:00,470 | WARNING | utils.py:337 | Cannot read edf file: /raid/sleepwalker/shhs/shhs/polysomnography/edfs/shhs1/shhs1-201276.edf due to [Errno 2] No such file or directory: '/raid/sleepwalker/shhs/shhs/polysomnography/annotations-events-profusion/shhs1/shhs1-201276-profusion.xml'


2025-10-17 13:51:00,474 | WARNING | utils.py:337 | Traceback (most recent call last):
  File "/root/projects/sleepwalker/sleepwalker/datasets/Basedataset.py", line 251, in prepare_patient
    df_additional = self.get_extra_event_df(edf_path, start)
  File "/root/projects/sleepwalker/sleepwalker/datasets/SHHS.py", line 63, in get_extra_event_df
    df = read_xml(fpath, "profusion", self.trim_wake, start_date)
  File "/root/projects/sleepwalker/sleepwalker/datasets/SHHS.py", line 18, in read_xml
    xml_df = read_profusion(xml_path)
  File "/root/projects/sleepwalker/sleepwalker/datasets/utils.py", line 201, in read_profusion
    def read_profusion(xml_path):
FileNotFoundError: [Errno 2] No such file or directory: '/raid/sleepwalker/shhs/shhs/polysomnography/annotations-events-profusion/shhs1/shhs1-201276-profusion.xml'



2025-10-17 13:26:15,204 | INFO | Preparing labels and sliding windows:   6%|▌         | 617/10481 [24:45<1:11:34,  2.30it/s]

read -1, less than 3960000 requested!!!


In [ ]:
# from sleepwalker.core.signal import fix_edf_header
# from sleepwalker.datasets.utils import get_edf_files_in_repo

# edf_files = get_edf_files_in_repo("/cephfs_projects/sleepwalker/hchs/polysomnography", recursive=True)
# print(f"Found {len(edf_files)} files")

# for e in edf_files:
#     fix_edf_header(e,dry=False)